In [1]:
# Select which function to analyse and create X and Y arrays


import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel, Matern, RationalQuadratic
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer
from scipy.stats import norm, qmc, rankdata
from scipy.spatial.distance import cdist
from itertools import product
import ast
import os
import re




#Select function to test
Func_to_test = 1

# Parameters
week = 13  # latest week to load

random_seed = 42

# ── PATH SETUP ──────────────────────────────────────────────
# Repo-relative paths. Assumes notebook is in notebooks/ (one level below root).
# If the notebook is in the repo root instead, change ".." to "."
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_dir  = os.path.join(repo_root, "data")

X_path = os.path.join(data_dir, "initial_data",
                      f"function_{Func_to_test}", "initial_inputs.npy")
Y_path = os.path.join(data_dir, "initial_data",
                      f"function_{Func_to_test}", "initial_outputs.npy")
# ────────────────────────────────────────────────────────────


# Load initial calibration data
X = np.load(X_path)
Y = np.load(Y_path)




# -----------------------
# CLEAN TEXT 
# -----------------------
def clean_text(text):
    # 1. Remove np.float64(...) FIRST (critical)
    text = re.sub(r'np\.float64\(([^()]*)\)', r'\1', text)

    # 2. Replace array([...]) → [...]
    text = re.sub(r'array\((\[[^\]]*\])\)', r'\1', text, flags=re.S)

    return text

# -----------------------
# SPLIT INTO TOP-LEVEL LISTS
# -----------------------
def split_top_level_lists(text):
    blocks = []
    bracket_count = 0
    current = ""

    for char in text:
        if char == '[':
            bracket_count += 1

        if bracket_count > 0:
            current += char

        if char == ']':
            bracket_count -= 1
            if bracket_count == 0:
                blocks.append(current)
                current = ""

    return blocks

# -----------------------
# LOAD FILES (FULL TEXT)
# -----------------------
week_path = os.path.join(data_dir, f"Week {week}")
with open(os.path.join(week_path, "inputs.txt"), "r") as f:
    inputs_text = f.read()
with open(os.path.join(week_path, "outputs.txt"), "r") as f:
    outputs_text = f.read()

# -----------------------
# PROCESS TEXT
# -----------------------
inputs_text = clean_text(inputs_text)
outputs_text = clean_text(outputs_text)

inputs_blocks = split_top_level_lists(inputs_text)
outputs_blocks = split_top_level_lists(outputs_text)

# Convert to Python objects
inputs_all = [ast.literal_eval(block) for block in inputs_blocks]
outputs_all = [ast.literal_eval(block) for block in outputs_blocks]


# -----------------------
# SELECT FUNCTION DATA
# -----------------------
idx = (Func_to_test - 1)

x_new_list = []
y_new_list = []

for batch_inputs, batch_outputs in zip(inputs_all, outputs_all):
    x_new_list.append(batch_inputs[idx])
    y_new_list.append(batch_outputs[idx])

x_new = np.array(x_new_list)
y_new = np.array(y_new_list)

# -----------------------
# APPEND DATA
# -----------------------
if x_new.shape[1] == X.shape[1]:
    X = np.vstack([X, x_new])
    Y = np.concatenate([Y, y_new])
else:
    print(f"Dimension mismatch: X is {X.shape[1]}D, new point is {x_new.shape[1]}D → skipped")


# Path to save CSVs
save_path = os.path.join(data_dir, "calibrated_data")
os.makedirs(save_path, exist_ok=True)

x_file = os.path.join(save_path, f"X_calibrated_func{Func_to_test}.csv")
y_file = os.path.join(save_path, f"Y_calibrated_func{Func_to_test}.csv")

np.savetxt(x_file, X, delimiter=",", comments='')
np.savetxt(y_file, Y.reshape(-1, 1), delimiter=",", comments='')

print(f"Saved X to: {x_file}")
print(f"Saved Y to: {y_file}")




# Print calibration data
print("\nX inputs\n",X)
print("\nY outputs\n",Y)
print("\nX shape:", X.shape)
print("Y shape:", Y.shape)
print("\nTesting function :", Func_to_test)

for k in range(week, 0, -1):
    value = Y[-k]
    
    # compare against all previous values up to that point
    prev_values = Y[:-k] if k != 1 else Y
    
    print(f"\nY value (index -{k}): {value}    max?: {value >= np.max(prev_values)}")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\data\\initial_data\\function_1\\initial_inputs.npy'

In [ ]:
"""
Bayesian optimisation pipeline for the BBO capstone.

Three optimisation modes:

  "standard"     : single-kernel GP + EI/UCB/PI on a global grid.
  "multikernel"  : Optuna-style multi-kernel Thompson sampling on a global grid.
                   Fits one GP per kernel, draws one posterior sample per
                   kernel per candidate, picks the best across all kernels.
  "turbo"        : Local BO inside an adaptive trust region.

Per-function defaults live in FUNC_CONFIG. Override MODE at the top to force
a single mode across all functions.

Each function in FUNC_CONFIG specifies:
  transform    : Y transform ("standardise", "log", "yeo_johnson", "signed_log")
  mode_hint    : "standard" | "multikernel" | "turbo"
  kernel_kind  : "matern15" | "matern25" | "rbf"  (used by standard/turbo)
  acq_name     : "ei" | "ucb" | "pi"  (used by standard mode)
  acq_param    : xi for ei/pi, beta for ucb
  percentile   : top-(100-p)% labelled "good" for SVM partition;
                 None disables the SVM filter
"""

import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel, Matern, RBF, WhiteKernel,
)
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from scipy.stats import norm, qmc
from scipy.spatial.distance import cdist


# ============================================================================
# Configuration
# ============================================================================

# --- Run-time choices --------------------------------------------------------
MODE                     = "auto"   # "auto" picks per-function from FUNC_CONFIG
                                    # or force "standard"/"multikernel"/"turbo"

# --- Grid construction -------------------------------------------------------
MAX_POINTS               = 20_000
USE_LATIN_HYPERCUBE      = False     # False → Sobol (better for d > 2)
EXCLUSION_RADIUS         = 0.005

# --- TuRBO -------------------------------------------------------------------
TR_BASE_LENGTH           = 0.5
TR_MIN_LENGTH            = 0.5 ** 7
TR_MAX_LENGTH            = 1.6
TR_SUCCESS_TOL           = 3
TR_FAILURE_TOL_FACTOR    = 1
USE_MULTIKERNEL_IN_TURBO = True

# --- Multi-kernel ------------------------------------------------------------
MULTIKERNEL_KINDS        = ("matern15", "matern25", "rbf")

# --- Other -------------------------------------------------------------------
RANDOM_SEED              = 42
N_RESTARTS_OPTIMIZER     = 10


# ============================================================================
# Per-function configuration
# ============================================================================
FUNC_CONFIG = {
    1: dict(transform="signed_log",  mode_hint="maximin_interior",
            kernel_kind="matern25", acq_name="ucb", acq_param=1,
            svm=dict(enabled=False, percentile=80, C=1.0, gamma="scale")),
    2: dict(transform="standardise", mode_hint="standard",
            kernel_kind="matern25", acq_name="ucb", acq_param=1,
            svm=dict(enabled=False, percentile=70, C=1.0, gamma="scale")),
    3: dict(transform="standardise", mode_hint="multikernel",
            kernel_kind="matern25", acq_name="ucb", acq_param=1,
            svm=dict(enabled=True, percentile=70, C=1.0, gamma="scale")),
    4: dict(transform="yeo_johnson", mode_hint="standard",
            kernel_kind="matern25", acq_name="ucb", acq_param=1,
            svm=dict(enabled=True, percentile=75, C=1.0, gamma="scale")),
    5: dict(transform="log",         mode_hint="standard",
            kernel_kind="matern25", acq_name="ucb", acq_param=1,
            svm=dict(enabled=True, percentile=75, C=1.0, gamma="scale")),
    6: dict(transform="standardise", mode_hint="multikernel",
            kernel_kind="matern25", acq_name="ucb", acq_param=1,
            svm=dict(enabled=True, percentile=70, C=1.0, gamma="scale")),
    7: dict(transform="standardise", mode_hint="standard",
            kernel_kind="matern25", acq_name="ucb", acq_param=1,
            svm=dict(enabled=True, percentile=75, C=1.0, gamma="scale")),
    8: dict(transform="standardise", mode_hint="turbo",
            kernel_kind="matern25", acq_name="ucb", acq_param=1,
            svm=dict(enabled=True, percentile=75, C=1.0, gamma="scale")),
}


# ============================================================================
# Pretty printing
# ============================================================================
def format_values(values):
    """Flatten, format to 6dp, join with hyphens — same convention as the
    rest of the project."""
    values_flat = np.asarray(values).flatten()
    formatted = [f"{v:.6f}" for v in values_flat]
    return "-".join(formatted)


# ============================================================================
# Y transformation
# ============================================================================
def transform_Y(Y, kind):
    Y = np.asarray(Y, dtype=float).ravel()

    if kind == "standardise":
        mu, sd = Y.mean(), Y.std() + 1e-12
        return (Y - mu) / sd

    if kind == "log":
        if np.any(Y <= 0):
            raise ValueError("log transform requires strictly positive Y.")
        Y_log = np.log(Y)
        return (Y_log - Y_log.mean()) / (Y_log.std() + 1e-12)

    if kind == "yeo_johnson":
        pt = PowerTransformer(method="yeo-johnson", standardize=True)
        return pt.fit_transform(Y.reshape(-1, 1)).ravel()

    if kind == "signed_log":
        floor = 1e-300
        Y_log = np.log10(np.maximum(Y, floor))
        return (Y_log - Y_log.mean()) / (Y_log.std() + 1e-12)

    raise ValueError(f"Unknown transform kind: {kind}")


# ============================================================================
# Kernel factory
# ============================================================================
def make_kernel(d, kind="matern25"):
    """ARD kernel with ConstantKernel signal variance and learnable WhiteKernel."""
    constant = ConstantKernel(1.0, (1e-4, 1e3))
    white    = WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1.0))
    ls       = [0.2] * d
    bounds   = (1e-2, 3.0)

    if kind == "matern25":
        body = Matern(length_scale=ls, length_scale_bounds=bounds, nu=2.5)
    elif kind == "matern15":
        body = Matern(length_scale=ls, length_scale_bounds=bounds, nu=1.5)
    elif kind == "rbf":
        body = RBF(length_scale=ls, length_scale_bounds=bounds)
    else:
        raise ValueError(f"Unknown kernel kind: {kind}")

    return constant * body + white


def fit_gp(X, Y, kernel, seed=RANDOM_SEED):
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=0.0,
        n_restarts_optimizer=N_RESTARTS_OPTIMIZER,
        random_state=seed,
    )
    gp.fit(X, Y)
    return gp


# ============================================================================
# Acquisition functions
# ============================================================================
def acquisition_ucb(mean, std, beta):
    return mean + beta * std

def acquisition_ei(mean, std, best_y, xi):
    std = np.maximum(std, 1e-9)
    z = (mean - best_y - xi) / std
    return (mean - best_y - xi) * norm.cdf(z) + std * norm.pdf(z)

def acquisition_pi(mean, std, best_y, xi):
    std = np.maximum(std, 1e-9)
    z = (mean - best_y - xi) / std
    return norm.cdf(z)

def compute_acquisition(name, mean, std, best_y, param):
    if name == "ucb":
        return acquisition_ucb(mean, std, beta=param)
    if name == "ei":
        return acquisition_ei(mean, std, best_y=best_y, xi=param)
    if name == "pi":
        return acquisition_pi(mean, std, best_y=best_y, xi=param)
    raise ValueError(f"Unknown acquisition: {name}")


# ============================================================================
# Grid construction
# ============================================================================
def build_grid(d, n_points=MAX_POINTS, lhs=USE_LATIN_HYPERCUBE, seed=RANDOM_SEED):
    if lhs:
        sampler = qmc.LatinHypercube(d=d, seed=seed)
        return sampler.random(n=n_points)
    sampler = qmc.Sobol(d=d, scramble=True, seed=seed)
    m = int(np.log2(n_points))
    return sampler.random_base2(m=m)


def build_local_grid(x_center, L, d, n_points, seed):
    sampler = qmc.Sobol(d=d, scramble=True, seed=seed)
    m = max(1, int(np.log2(n_points)))
    box  = sampler.random_base2(m=m) - 0.5
    pts  = x_center + L * box
    return np.clip(pts, 0.0, 1.0)


# ============================================================================
# SVM partition filter (JetBrains-style)
# ============================================================================
def svm_partition_mask(X, Y_t, x_grid, svm_cfg):
    """
    Train an SVM to predict whether a point is in the top-(100-percentile)%
    of Y, then return a boolean mask over x_grid (True = SVM thinks 'good').

    svm_cfg dict keys:
      enabled    : bool, master on/off switch
      percentile : float, top-(100-p)% labelled positive
      C          : float, SVM regularisation strength (default 1.0)
      gamma      : str or float, RBF gamma (default "scale")
    """
    info = {"enabled":     None,
            "percentile":  None,
            "C":           None,
            "gamma":       None,
            "threshold":   None,
            "n_positive":  None,
            "frac_kept":   None,
            "status":      "applied"}

    if svm_cfg is None or not svm_cfg.get("enabled", False):
        info["enabled"]   = False
        info["status"]    = "disabled"
        info["frac_kept"] = 1.0
        return np.ones(len(x_grid), dtype=bool), info

    percentile = svm_cfg["percentile"]
    C          = svm_cfg.get("C", 1.0)
    gamma      = svm_cfg.get("gamma", "scale")

    info.update({"enabled": True, "percentile": percentile,
                 "C": C, "gamma": gamma})

    threshold = np.percentile(Y_t, percentile)
    labels    = (Y_t >= threshold).astype(int)
    n_pos, n_neg = int(labels.sum()), int((1 - labels).sum())

    info["threshold"]  = float(threshold)
    info["n_positive"] = n_pos

    if n_pos < 3 or n_neg < 3:
        info["status"]    = f"skipped (n_pos={n_pos}, n_neg={n_neg})"
        info["frac_kept"] = 1.0
        return np.ones(len(x_grid), dtype=bool), info

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("svm",    SVC(kernel="rbf", class_weight="balanced",
                       C=C, gamma=gamma)),
    ])
    pipe.fit(X, labels)
    mask = pipe.decision_function(x_grid) > 0
    frac = float(mask.mean())
    info["frac_kept"] = frac

    if frac < 0.005 or frac > 0.995:
        info["status"]    = f"degenerate (frac={frac:.3f}); bypassed"
        return np.ones(len(x_grid), dtype=bool), info

    return mask, info

  


# ============================================================================
# Mode 1: Standard GP-BO
# ============================================================================
def propose_standard(X, Y_t, x_grid, d, cfg):
    gp = fit_gp(X, Y_t, make_kernel(d, cfg["kernel_kind"]))
    mean, std = gp.predict(x_grid, return_std=True)
    acq = compute_acquisition(cfg["acq_name"], mean, std,
                              best_y=np.max(Y_t), param=cfg["acq_param"])

    # Apply SVM partition mask
    mask, mask_info = svm_partition_mask(X, Y_t, x_grid, cfg.get("svm"))
    acq[~mask] = -np.inf

    # Exclusion radius around existing observations
    min_d = cdist(x_grid, X).min(axis=1)
    acq[min_d < EXCLUSION_RADIUS] = -np.inf

    if not np.isfinite(acq).any():
        # Both filters wiped everything — fall back to maximin
        idx = int(np.argmax(min_d))
        diag_note = "all candidates filtered; fell back to maximin"
    else:
        idx = int(np.argmax(acq))
        diag_note = "ok"

    return {
        "x_next": x_grid[idx:idx+1],
        "diag":   {"acq_name":       cfg["acq_name"],
                   "acq_param":      cfg["acq_param"],
                   "kernel_kind":    cfg["kernel_kind"],
                   "fitted_kernel":  str(gp.kernel_),
                   "post_std_range": (float(std.min()), float(std.max())),
                   "max_acq":        float(acq[idx]) if np.isfinite(acq[idx]) else None,
                   "svm_filter":     mask_info,
                   "note":           diag_note},
    }


# ============================================================================
# Mode 2: Multi-kernel Thompson sampling
# ============================================================================
def propose_multikernel(X, Y_t, x_grid, d, cfg, kernel_kinds=MULTIKERNEL_KINDS,
                        seed=RANDOM_SEED):
    """
    Fit one GP per kernel; draw one posterior sample per candidate per kernel;
    pick the candidate with the largest sample across all kernels.
    The SVM partition mask (if configured) and exclusion radius apply once,
    consistently across all kernels.
    """
    rng  = np.random.default_rng(seed)

    mask, mask_info = svm_partition_mask(X, Y_t, x_grid, cfg.get("svm"))
    excluded = (cdist(x_grid, X).min(axis=1) < EXCLUSION_RADIUS) | (~mask)

    best = {"sample": -np.inf, "x": None, "kernel": None}
    per_kernel = []

    for kind in kernel_kinds:
        gp = fit_gp(X, Y_t, make_kernel(d, kind), seed)
        mean, std = gp.predict(x_grid, return_std=True)
        sample    = mean + std * rng.standard_normal(len(x_grid))
        sample[excluded] = -np.inf

        if not np.isfinite(sample).any():
            per_kernel.append({"kind": kind, "fitted_kernel": str(gp.kernel_),
                               "best_sample": None})
            continue

        idx = int(np.argmax(sample))
        per_kernel.append({"kind": kind,
                           "fitted_kernel": str(gp.kernel_),
                           "best_sample":   float(sample[idx])})
        if sample[idx] > best["sample"]:
            best = {"sample": float(sample[idx]),
                    "x":      x_grid[idx:idx+1],
                    "kernel": kind}

    if best["x"] is None:
        # Total filter wipeout — fall back to maximin over the full grid
        min_d = cdist(x_grid, X).min(axis=1)
        idx   = int(np.argmax(min_d))
        return {
            "x_next": x_grid[idx:idx+1],
            "diag":   {"chosen_kernel": None, "best_sample": None,
                       "per_kernel":   per_kernel,
                       "svm_filter":   mask_info,
                       "note":         "filter wipeout; maximin fallback"},
        }

    return {
        "x_next": best["x"],
        "diag":   {"chosen_kernel": best["kernel"],
                   "best_sample":   best["sample"],
                   "per_kernel":    per_kernel,
                   "svm_filter":    mask_info},
    }


# ============================================================================
# Mode 3: TuRBO
# ============================================================================
def compute_trust_region_length(Y, d,
                                base=TR_BASE_LENGTH,
                                L_min=TR_MIN_LENGTH, L_max=TR_MAX_LENGTH,
                                success_tol=TR_SUCCESS_TOL,
                                failure_tol_factor=TR_FAILURE_TOL_FACTOR):
    """Replay the Y history and adapt L using TuRBO success/failure rule."""
    failure_tol = max(4, failure_tol_factor * d)
    n_init      = max(2, d + 1)

    L = base
    cumulative_max = -np.inf
    successes = failures = 0

    for i, y in enumerate(Y):
        if i < n_init:
            cumulative_max = max(cumulative_max, y)
            continue
        if y > cumulative_max:
            successes += 1; failures = 0
            cumulative_max = y
        else:
            failures += 1; successes = 0
        if successes == success_tol:
            L = min(L * 2, L_max);  successes = 0
        if failures == failure_tol:
            L = max(L / 2, L_min / 2);  failures = 0

    return L, (L < L_min)


def propose_turbo(X, Y_t, d, cfg, multikernel=USE_MULTIKERNEL_IN_TURBO,
                  seed=RANDOM_SEED):
    """
    TuRBO-1: local BO inside an adaptive trust region. With multikernel=True,
    uses Optuna-style multi-kernel Thompson sampling inside the TR.
    Restart trigger: TR collapses below TR_MIN_LENGTH → fall back to global
    Sobol maximin selection.
    """
    rng = np.random.default_rng(seed)
    L, restart = compute_trust_region_length(Y_t, d=d)

    if restart:
        candidates = build_grid(d, MAX_POINTS, lhs=False, seed=seed)
        min_d      = cdist(candidates, X).min(axis=1)
        idx        = int(np.argmax(min_d))
        return {
            "x_next": candidates[idx:idx+1],
            "diag":   {"trust_region_L": float(L), "restart": True,
                       "note": "TR collapsed; global Sobol maximin restart",
                       "maximin_distance": float(min_d[idx])},
        }

    x_center   = X[int(np.argmax(Y_t))]
    candidates = build_local_grid(x_center, L, d, MAX_POINTS, seed)
    excluded   = cdist(candidates, X).min(axis=1) < EXCLUSION_RADIUS

    if multikernel:
        kernel_kinds = MULTIKERNEL_KINDS
    else:
        kernel_kinds = (cfg["kernel_kind"],)

    best = {"sample": -np.inf, "x": None, "kernel": None}

    for kind in kernel_kinds:
        gp = fit_gp(X, Y_t, make_kernel(d, kind), seed)
        mean, std = gp.predict(candidates, return_std=True)
        sample    = mean + std * rng.standard_normal(len(candidates))
        sample[excluded] = -np.inf

        idx = int(np.argmax(sample))
        if sample[idx] > best["sample"]:
            best = {"sample": float(sample[idx]),
                    "x":      candidates[idx:idx+1],
                    "kernel": kind}

    return {
        "x_next": best["x"],
        "diag":   {"trust_region_L":  float(L),
                   "x_center":        x_center.tolist(),
                   "chosen_kernel":   best["kernel"],
                   "best_sample":     best["sample"],
                   "multikernel":     multikernel,
                   "n_kernels_tried": len(kernel_kinds)},
    }


# ============================================================================
# Main dispatcher
# ============================================================================
def propose_next(X, Y, func_id, mode_override=None, verbose=True):
    if func_id not in FUNC_CONFIG:
        raise KeyError(f"No FUNC_CONFIG entry for function {func_id}")

    cfg  = FUNC_CONFIG[func_id]
    d    = X.shape[1]
    mode = mode_override or cfg["mode_hint"]
    Y_t  = transform_Y(Y, cfg["transform"])

    if verbose:
        print(f"\n=== Function {func_id}  (d={d}, n={len(Y)}) ===")
        print(f"  transform   : {cfg['transform']:<14}  "
              f"Y_t mean={Y_t.mean():+.3f}, std={Y_t.std():.3f}, "
              f"min={Y_t.min():+.3f}, max={Y_t.max():+.3f}")
        print(f"  mode        : {mode}")
        print(f"  kernel_kind : {cfg['kernel_kind']}")
        if mode == "standard":
            param_label = "β" if cfg["acq_name"] == "ucb" else "ξ"
            print(f"  acquisition : {cfg['acq_name'].upper():<3}   "
                  f"{param_label}={cfg['acq_param']}")
        elif mode == "multikernel":
            print(f"  acquisition : Thompson sampling across "
                  f"{MULTIKERNEL_KINDS}")
        elif mode == "turbo":
            print(f"  acquisition : Thompson sampling in trust region "
                  f"(multikernel={USE_MULTIKERNEL_IN_TURBO})")
        svm_cfg = cfg.get("svm") or {}
        if svm_cfg.get("enabled"):
            print(f"  svm filter  : enabled "
                f"(percentile={svm_cfg.get('percentile')}, "
                f"C={svm_cfg.get('C', 1.0)}, gamma={svm_cfg.get('gamma', 'scale')!r})")
        else:
            print(f"  svm filter  : disabled")

    if mode == "standard":
        x_grid = build_grid(d, MAX_POINTS, USE_LATIN_HYPERCUBE, RANDOM_SEED)
        result = propose_standard(X, Y_t, x_grid, d, cfg)
    elif mode == "multikernel":
        x_grid = build_grid(d, MAX_POINTS, USE_LATIN_HYPERCUBE, RANDOM_SEED)
        result = propose_multikernel(X, Y_t, x_grid, d, cfg)
    elif mode == "turbo":
        result = propose_turbo(X, Y_t, d, cfg,
                               multikernel=USE_MULTIKERNEL_IN_TURBO)
    elif mode == "maximin":
        result = propose_maximin(X, Y_t, d)
    elif mode == "maximin_interior":
        result = propose_maximin_interior(X, d)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    x_next = result["x_next"]
    dist   = float(cdist(x_next, X).min())

    if verbose:
        x_best  = X[int(np.argmax(Y_t))]
        y_best  = Y[int(np.argmax(Y_t))]
        print(f"  x_best      : {format_values(x_best)}    (Y_raw={y_best:.4g})")
        print(f"  x_next      : {format_values(x_next)}    "
              f"(nearest-train dist = {dist:.4f})")
        for k, v in result["diag"].items():
            if isinstance(v, dict):
                print(f"     {k}:")
                for kk, vv in v.items():
                    print(f"        {kk}: {vv}")
            else:
                print(f"     {k}: {v}")

    return x_next, result


def propose_maximin(X, Y_t, d, seed=RANDOM_SEED):
    sampler = qmc.Sobol(d=d, scramble=True, seed=seed)
    candidates = sampler.random(n=2048)
    min_dists = cdist(candidates, X).min(axis=1)
    idx = int(np.argmax(min_dists))
    return {
        "x_next": candidates[idx:idx+1],
        "diag": {"strategy": "Sobol maximin coverage",
                 "n_candidates": len(candidates),
                 "maximin_distance": float(min_dists[idx])},
    }



def propose_maximin_interior(X, d, seed=42, interior_weight=0.5):
    sampler = qmc.Sobol(d=d, scramble=True, seed=seed)
    cand = sampler.random(n=2048)
    md = cdist(cand, X).min(axis=1)
    interior = np.prod(cand * (1 - cand), axis=1) ** (1.0 / (2 * d))
    interior /= interior.max()
    score = md * ((1 - interior_weight) + interior_weight * interior)
    idx = int(np.argmax(score))
    return {
        "x_next": cand[idx:idx+1],
        "diag": {"strategy": f"interior-biased maximin (w={interior_weight})",
                 "maximin_distance": float(md[idx]),
                 "interior_factor": float(interior[idx])},
    }



# ============================================================================
# Run on the currently-loaded function
# ============================================================================
for _name in ("X", "Y", "Func_to_test"):
    if _name not in globals():
        raise NameError(f"`{_name}` is not defined — run the data-loading cell first.")

x_next, result = propose_next(
    X, Y, Func_to_test,
    mode_override=(None if MODE == "auto" else MODE),
)

In [ ]:
# ============================================================================
# 2D / 3D plotting (2D inputs only)
# ============================================================================
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)


def _build_meshgrid(n=100):
    """Return (pts, X1, X2): flat (n*n, 2) array and the two meshgrid arrays."""
    x1 = np.linspace(0, 1, n)
    x2 = np.linspace(0, 1, n)
    X1, X2 = np.meshgrid(x1, x2)
    pts = np.column_stack([X1.ravel(), X2.ravel()])
    return pts, X1, X2


def _draw_tr_box(ax, x_center, L):
    """Draw the TuRBO trust-region rectangle on a 2D axes."""
    x0 = max(0.0, x_center[0] - L / 2)
    y0 = max(0.0, x_center[1] - L / 2)
    w  = min(1.0, x_center[0] + L / 2) - x0
    h  = min(1.0, x_center[1] + L / 2) - y0
    ax.add_patch(Rectangle((x0, y0), w, h, fill=False,
                           edgecolor="white", linewidth=2,
                           linestyle="--", label=f"TR (L={L:.2f})"))


def _panel_2d(ax, X1, X2, Z, X, x_best, x_next, title, cmap,
              cbar_label, tr_box=None, contour_fmt="%.2f",
              extra_scatter=None):
    cf = ax.contourf(X1, X2, Z, levels=50, cmap=cmap)
    cs = ax.contour(X1, X2, Z, levels=8, colors="black",
                    linewidths=0.5, alpha=0.6)
    ax.clabel(cs, inline=True, fontsize=8, fmt=contour_fmt)

    ax.scatter(X[:, 0], X[:, 1], c="red", s=40,
               edgecolor="black", linewidths=0.5, label="Observed", zorder=4)
    ax.scatter(*x_best, c="orange", s=160, edgecolor="black",
               marker="o", label="Best", zorder=5)
    if x_next is not None:
        ax.scatter(*np.asarray(x_next).flatten(), c="yellow", s=240,
                   edgecolor="black", marker="*", label="x_next", zorder=6)
    if tr_box is not None:
        _draw_tr_box(ax, *tr_box)
    if extra_scatter is not None:
        for kwargs in extra_scatter:
            ax.scatter(**kwargs)

    ax.set_title(title)
    ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.set_xlim(0, 1);   ax.set_ylim(0, 1)
    ax.legend(loc="upper right", fontsize=8, framealpha=0.85)

    cbar = plt.colorbar(cf, ax=ax)
    cbar.ax.tick_params(labelsize=9)
    cbar.set_label(cbar_label, rotation=270, labelpad=15)


def _panel_3d(ax, X1, X2, Z, X_obs, Y_obs, title, cmap, zlabel):
    ax.plot_surface(X1, X2, Z, cmap=cmap, alpha=0.75,
                    linewidth=0, antialiased=True)
    if X_obs is not None and Y_obs is not None:
        ax.scatter(X_obs[:, 0], X_obs[:, 1], Y_obs,
                   c="red", s=40, edgecolor="black", linewidths=0.5,
                   label="Observed")
        ax.legend(loc="upper right", fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.set_zlabel(zlabel)
    ax.view_init(elev=10, azim=-145)

# ----------------------------------------------------------------------------
# GP-mean and acquisition surfaces
# ----------------------------------------------------------------------------
def plot_gp_2d(X, Y, func_id, x_next=None, kind="2d", n_plot=100):
    """
    Visualise GP mean and acquisition surfaces for 2D functions.

    kind = "2d"   → heatmap pair
           "3d"   → 3D-surface pair
           "both" → 2x2 layout (heatmaps top, surfaces bottom)
    """
    if X.shape[1] != 2:
        print(f"plot_gp_2d: skipping — requires d=2, got d={X.shape[1]}")
        return

    cfg = FUNC_CONFIG[func_id]
    Y_t = transform_Y(Y, cfg["transform"])

    pts, X1, X2 = _build_meshgrid(n_plot)
    gp = fit_gp(X, Y_t, make_kernel(d=2, kind=cfg["kernel_kind"]))
    mean, std = gp.predict(pts, return_std=True)
    acq = compute_acquisition(cfg["acq_name"], mean, std,
                              best_y=np.max(Y_t), param=cfg["acq_param"])

    Z_mean = mean.reshape(X1.shape)
    Z_acq  = acq.reshape(X1.shape)

    best_idx = int(np.argmax(Y_t))
    x_best   = X[best_idx]

    # TuRBO trust-region overlay
    tr_box = None
    if cfg["mode_hint"] == "turbo":
        L, restart = compute_trust_region_length(Y_t, d=2)
        if not restart:
            tr_box = (x_best, L)

    param_label = "β" if cfg["acq_name"] == "ucb" else "ξ"
    title_mean  = f"Func {func_id}: GP mean ({cfg['kernel_kind']}, transform={cfg['transform']})"
    title_acq   = (f"Func {func_id}: {cfg['acq_name'].upper()} acquisition "
                   f"({param_label}={cfg['acq_param']})")
    acq_label   = cfg['acq_name'].upper()

    if kind == "2d":
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        _panel_2d(axes[0], X1, X2, Z_mean, X, x_best, x_next,
                  title_mean, "viridis", "GP mean", tr_box=tr_box)
        _panel_2d(axes[1], X1, X2, Z_acq, X, x_best, x_next,
                  title_acq, "magma", acq_label, tr_box=tr_box)

    elif kind == "3d":
        fig = plt.figure(figsize=(13, 5))
        ax1 = fig.add_subplot(1, 2, 1, projection="3d")
        ax2 = fig.add_subplot(1, 2, 2, projection="3d")
        _panel_3d(ax1, X1, X2, Z_mean, X, Y_t, title_mean, "viridis", "Y_t")
        _panel_3d(ax2, X1, X2, Z_acq,  None, None, title_acq, "magma", acq_label)

    elif kind == "both":
        fig = plt.figure(figsize=(13, 10))
        ax_h_mean = fig.add_subplot(2, 2, 1)
        ax_h_acq  = fig.add_subplot(2, 2, 2)
        ax_s_mean = fig.add_subplot(2, 2, 3, projection="3d")
        ax_s_acq  = fig.add_subplot(2, 2, 4, projection="3d")
        _panel_2d(ax_h_mean, X1, X2, Z_mean, X, x_best, x_next,
                  title_mean, "viridis", "GP mean", tr_box=tr_box)
        _panel_2d(ax_h_acq, X1, X2, Z_acq, X, x_best, x_next,
                  title_acq, "magma", acq_label, tr_box=tr_box)
        _panel_3d(ax_s_mean, X1, X2, Z_mean, X, Y_t, title_mean, "viridis", "Y_t")
        _panel_3d(ax_s_acq,  X1, X2, Z_acq,  None, None, title_acq, "magma", acq_label)

    else:
        raise ValueError(f"Unknown kind: {kind!r}; use '2d', '3d', or 'both'")

    plt.tight_layout()
    plt.show()

    print(f"Function {func_id}, fitted kernel: {gp.kernel_}")
    print(f"Posterior std range: ({std.min():.3f}, {std.max():.3f})")
    print(f"Acquisition: {cfg['acq_name'].upper()}, "
          f"max acq value over plot grid: {acq.max():.4g}")


# ----------------------------------------------------------------------------
# SVM partition surfaces
# ----------------------------------------------------------------------------
def _fit_svm_for_plot(X, Y_t, svm_cfg):
    if svm_cfg is None or not svm_cfg.get("enabled", False):
        return None, None, None, "disabled"

    percentile = svm_cfg["percentile"]
    C          = svm_cfg.get("C", 1.0)
    gamma      = svm_cfg.get("gamma", "scale")

    threshold = np.percentile(Y_t, percentile)
    labels    = (Y_t >= threshold).astype(int)
    n_pos, n_neg = int(labels.sum()), int((1 - labels).sum())
    if n_pos < 3 or n_neg < 3:
        return None, labels, threshold, f"untrainable (n_pos={n_pos}, n_neg={n_neg})"

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("svm",    SVC(kernel="rbf", class_weight="balanced",
                       C=C, gamma=gamma)),
    ])
    pipe.fit(X, labels)
    return pipe, labels, threshold, "trained"


def plot_svm_partition_2d(X, Y, func_id, x_next=None, kind="2d", n_plot=100):
    """
    Visualise the SVM partition for 2D functions.

    Heatmap shows the SVM decision function (signed distance from boundary):
      positive = "in good region"  (mask kept)
      negative = "in bad region"   (mask rejected)

    Markers:
      red circle      = training point, low-Y class (label=0)
      green triangle  = training point, high-Y class (label=1)
      black square    = support vector
      orange circle   = best observed
      yellow star     = x_next

    kind = "2d" | "3d" | "both" — same convention as plot_gp_2d.
    """
    if X.shape[1] != 2:
        print(f"plot_svm_partition_2d: skipping — requires d=2, got d={X.shape[1]}")
        return

    cfg = FUNC_CONFIG[func_id]
    Y_t = transform_Y(Y, cfg["transform"])

    svm_cfg = cfg.get("svm", {}) or {}
    pipe, labels, threshold, status = _fit_svm_for_plot(X, Y_t, svm_cfg)
    if pipe is None:
        print(f"plot_svm_partition_2d: SVM not available — {status}")
        return

    pts, X1, X2 = _build_meshgrid(n_plot)
    decision = pipe.decision_function(pts)
    Z = decision.reshape(X1.shape)

    svm_step     = pipe.named_steps["svm"]
    sv_indices   = svm_step.support_
    sv_points    = X[sv_indices]
    n_pos, n_neg = int(labels.sum()), int((1 - labels).sum())
    frac_kept    = float((decision > 0).mean())

    best_idx = int(np.argmax(Y_t))
    x_best   = X[best_idx]

    pos = labels == 1

    percentile = svm_cfg.get("percentile", "?")
    C_value    = svm_cfg.get("C", 1.0)

    title = (f"Func {func_id}: SVM partition  "
             f"(percentile={percentile}, C={C_value}, "
             f"threshold Y_t={threshold:.3f})")
    cbar_label = "SVM decision (>0 = good region)"

    if kind in ("2d", "both"):
        if kind == "2d":
            fig, ax = plt.subplots(1, 1, figsize=(8, 6.5))
        else:
            fig = plt.figure(figsize=(13, 6))
            ax  = fig.add_subplot(1, 2, 1)

        cf = ax.contourf(X1, X2, Z, levels=50, cmap="RdBu_r")
        ax.contour(X1, X2, Z, levels=[0.0], colors="white",
                   linewidths=2.0, linestyles="-")
        cs = ax.contour(X1, X2, Z, levels=8, colors="black",
                        linewidths=0.4, alpha=0.5)
        ax.clabel(cs, inline=True, fontsize=8, fmt="%.2f")

        ax.scatter(X[~pos, 0], X[~pos, 1], c="red", s=50,
                   edgecolor="black", linewidths=0.6,
                   label=f"low Y (n={n_neg})", zorder=4)
        ax.scatter(X[pos, 0], X[pos, 1], c="lime", marker="^", s=80,
                   edgecolor="black", linewidths=0.6,
                   label=f"high Y (n={n_pos})", zorder=4)
        ax.scatter(sv_points[:, 0], sv_points[:, 1],
                   facecolors="none", edgecolors="black", marker="s",
                   s=180, linewidths=1.5,
                   label=f"support vector ({len(sv_points)})", zorder=5)
        ax.scatter(*x_best, c="orange", s=160, edgecolor="black",
                   marker="o", label="Best", zorder=6)
        if x_next is not None:
            ax.scatter(*np.asarray(x_next).flatten(), c="yellow", s=240,
                       edgecolor="black", marker="*", label="x_next", zorder=7)

        ax.set_title(title)
        ax.set_xlabel("x1")
        ax.set_ylabel("x2")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.legend(loc="upper right", fontsize=8, framealpha=0.85)

        cbar = plt.colorbar(cf, ax=ax)
        cbar.ax.tick_params(labelsize=9)
        cbar.set_label(cbar_label, rotation=270, labelpad=15)

        if kind == "both":
            ax3 = fig.add_subplot(1, 2, 2, projection="3d")
            _plot_svm_3d_panel(ax3, X1, X2, Z, X, labels, sv_points,
                               title=f"Func {func_id}: SVM decision surface")
        plt.tight_layout()
        plt.show()

    elif kind == "3d":
        fig = plt.figure(figsize=(8, 6.5))
        ax  = fig.add_subplot(1, 1, 1, projection="3d")
        _plot_svm_3d_panel(ax, X1, X2, Z, X, labels, sv_points,
                           title=f"Func {func_id}: SVM decision surface")
        plt.tight_layout()
        plt.show()

    else:
        raise ValueError(f"Unknown kind: {kind!r}; use '2d', '3d', or 'both'")

    print(f"\nFunction {func_id} SVM partition:")
    print(f"  status         : {status}")
    print(f"  Y_t threshold  : {threshold:.3f}  ({percentile}th percentile)")
    print(f"  class balance  : {n_pos} positive, {n_neg} negative")
    print(f"  support vectors: {len(sv_points)} (indices {sv_indices.tolist()})")
    print(f"  fraction kept  : {frac_kept:.1%} of plot grid")
   


def _plot_svm_3d_panel(ax, X1, X2, Z, X_obs, labels, sv_points, title):
    """3D surface of SVM decision function with a coloured zero plane."""
    ax.plot_surface(X1, X2, Z, cmap="RdBu_r", alpha=0.7,
                    linewidth=0, antialiased=True)
    # Zero plane (decision boundary)
    zero_plane = np.zeros_like(Z)
    ax.plot_surface(X1, X2, zero_plane, color="white", alpha=0.15,
                    linewidth=0, antialiased=False)

    pos = labels == 1
    # Plot training points at their decision-function value
    ax.scatter(X_obs[~pos, 0], X_obs[~pos, 1], np.zeros((~pos).sum()),
               c="red", s=40, edgecolor="black", linewidths=0.5,
               label="low Y")
    ax.scatter(X_obs[pos, 0], X_obs[pos, 1], np.zeros(pos.sum()),
               c="lime", marker="^", s=70, edgecolor="black", linewidths=0.5,
               label="high Y")
    ax.scatter(sv_points[:, 0], sv_points[:, 1], np.zeros(len(sv_points)),
               facecolors="none", edgecolors="black", marker="s",
               s=140, linewidths=1.2, label="support vector")

    ax.set_title(title)
    ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.set_zlabel("SVM decision")
    ax.legend(loc="upper right", fontsize=8)


# ============================================================================
# Usage with current function
# ============================================================================
if X.shape[1] == 2:
    plot_gp_2d(X, Y, Func_to_test, x_next=x_next, kind="both")
    plot_svm_partition_2d(X, Y, Func_to_test, x_next=x_next, kind="both")

In [ ]:
# ============================================================================
# Iterate through all functions and produce a consolidated report
# ============================================================================
import io
from contextlib import redirect_stdout
import warnings
from sklearn.exceptions import ConvergenceWarning





def _format_diag(diag, indent=5):
    """Recursive flat-text formatter for a diagnostic dict."""
    pad = " " * indent
    lines = []
    for k, v in diag.items():
        if isinstance(v, dict):
            lines.append(f"{pad}{k}:")
            lines.extend(_format_diag(v, indent + 3))
        elif isinstance(v, list) and v and isinstance(v[0], dict):
            lines.append(f"{pad}{k}:")
            for i, item in enumerate(v):
                lines.append(f"{pad}   [{i}]")
                lines.extend(_format_diag(item, indent + 6))
        else:
            lines.append(f"{pad}{k}: {v}")
    return lines


def run_all_functions(data_loader, func_ids=range(1, 9), mode_override=None):
    """
    Run propose_next for every function in func_ids and return a single
    consolidated text report.

    data_loader(func_id) -> (X, Y) tuple with the calibrated data for that
    function. Implementation depends on how your project organises files;
    a simple version is given below.
    """
    sections = []
    summary  = []

    for func_id in func_ids:
        try:
            X_f, Y_f = data_loader(func_id)
        except Exception as e:
            sections.append(f"\n=== Function {func_id} === FAILED TO LOAD: {e}\n")
            summary.append((func_id, None, None, None, f"load error: {e}"))
            continue

        try:
            # Capture verbose output from propose_next so it goes into the
            # consolidated report rather than scattered across cell output.
            buf = io.StringIO()
            with redirect_stdout(buf):
                x_next, result = propose_next(
                    X_f, Y_f, func_id,
                    mode_override=mode_override,
                    verbose=True,
                )
            sections.append(buf.getvalue().rstrip())

            # Add the formatted diagnostic block (already in the captured
            # output — but we also build a one-liner summary for the table)
            cfg     = FUNC_CONFIG[func_id]
            best_y  = float(np.max(Y_f))
            x_best  = X_f[int(np.argmax(Y_f))]
            x_next_str = format_values(x_next)
            summary.append((func_id, cfg["mode_hint"], best_y,
                            x_next_str, "ok"))

        except Exception as e:
            sections.append(f"\n=== Function {func_id} === FAILED: {e}\n")
            summary.append((func_id, None, None, None, f"error: {e}"))

    # Build the summary table
    table  = ["", "=" * 78,
              "SUMMARY", "=" * 78,
              f"{'Func':<6}{'Mode':<14}{'Best Y':<14}{'x_next':<60}{'Status'}"]
    for fid, mode, best_y, x_next_str, status in summary:
        table.append(f"{fid:<6}"
                     f"{(mode or '-'):<14}"
                     f"{(f'{best_y:.4g}' if best_y is not None else '-'):<14}"
                     f"{(x_next_str or '-'):<60}"
                     f"{status}")
    table.append("=" * 78)

    return "\n".join(sections + table)


# ----------------------------------------------------------------------------
# Data loader — adjust paths to match  project layout
# ----------------------------------------------------------------------------
def load_calibrated(func_id, base_path=None):
    """
    Load the calibrated CSVs your data pipeline writes out.
    Adjust the path template to match where the loading script saves files.
    """
    if base_path is None:
        base_path = os.path.join(data_dir, "calibrated_data")
    X_path = rf"{base_path}\X_calibrated_func{func_id}.csv"
    Y_path = rf"{base_path}\Y_calibrated_func{func_id}.csv"
    X_f = np.loadtxt(X_path, delimiter=",", ndmin=2)
    Y_f = np.loadtxt(Y_path, delimiter=",").ravel()
    return X_f, Y_f


# ----------------------------------------------------------------------------
# Run it
# ----------------------------------------------------------------------------
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    
    report = run_all_functions(
    data_loader=load_calibrated,
    func_ids=range(1, 9),
    mode_override=(None if MODE == "auto" else MODE),
)
print(report)

In [ ]:
"""
Summarise BBO progress per function.
 
Produces:
  1. A per-function summary table (current best, where/when found, improvement).
  2. A round-by-round log flagging which evaluations set a new maximum.
 
Set INITIAL_COUNTS to the number of initial-calibration points per function
(the rows that pre-date round 1).
"""
 
import numpy as np
import pandas as pd
import glob
import re
import sys

 
# Number of initial calibration points per function (rows before round 1).
INITIAL_COUNTS = {1: 10, 2: 10, 3: 15, 4: 30, 5: 20, 6: 20, 7: 30, 8: 40}


 
def load_all(directory, file_glob="*.csv"):
    """Load X/Y CSVs into {func_id: (X, Y)}. Uses the most recent file per
    (kind, func) if duplicate timestamps exist."""
    paths = glob.glob(f"{directory}/{file_glob}")
    latest = {}
    for p in sorted(paths):                      # sorted → later timestamp wins
        m = re.search(r"([XY])_calibrated_func(\d+)", p)
        if m:
            latest[(m.group(1), int(m.group(2)))] = p
    data = {}
    for (kind, fid), p in latest.items():
        arr = pd.read_csv(p, header=None).values
        data.setdefault(fid, {})[kind] = arr
    return {fid: (d["X"], d["Y"].ravel()) for fid, d in data.items() if "X" in d and "Y" in d}
 
 
def fmt_x(x, dp=4):
    return "(" + ", ".join(f"{v:.{dp}f}" for v in np.asarray(x).ravel()) + ")"
 
 
def summary_table(data, initial_counts):
    """Per-function: dimensionality, totals, current best, when found, improvement."""
    rows = []
    for fid in sorted(data):
        X, Y = data[fid]
        n = len(Y)
        n_init = initial_counts.get(fid, 0)
        n_rounds = n - n_init
 
        best_idx = int(np.argmax(Y))
        best_y = Y[best_idx]
 
        # Which round set the current best? (0 = found in initial data)
        if best_idx < n_init:
            best_round = 0
        else:
            best_round = best_idx - n_init + 1
 
        # Best within the initial data only, to measure improvement from rounds
        best_initial = np.max(Y[:n_init]) if n_init > 0 else np.nan
        improvement = best_y - best_initial if n_init > 0 else np.nan
 
        # How many rounds since the best was last improved?
        rounds_since_best = (n - 1 - best_idx)
 
        rows.append({
            "func": fid,
            "d": X.shape[1],
            "n_total": n,
            "n_rounds": n_rounds,
            "best_Y": best_y,
            "best_found_round": best_round,
            "best_initial": best_initial,
            "improvement": improvement,
            "rounds_since_best": rounds_since_best,
        })
    return pd.DataFrame(rows)
 
 
def new_max_log(data, initial_counts):
    """Round-by-round: every round evaluation, flagging new maxima."""
    rows = []
    for fid in sorted(data):
        X, Y = data[fid]
        n_init = initial_counts.get(fid, 0)
        running_best = np.max(Y[:n_init]) if n_init > 0 else -np.inf
 
        for i in range(n_init, len(Y)):
            rnd = i - n_init + 1
            y = Y[i]
            is_new_max = y > running_best
            improvement = (y - running_best) if is_new_max else 0.0
            if is_new_max:
                running_best = y
            rows.append({
                "func": fid,
                "round": rnd,
                "Y": y,
                "running_best": running_best,
                "new_max": "** NEW MAX **" if is_new_max else "",
                "gain": improvement if is_new_max else np.nan,
                "x": fmt_x(X[i]),
            })
    return pd.DataFrame(rows)
 
 
def print_report(directory, initial_counts=INITIAL_COUNTS, file_glob="*.csv"):
    data = load_all(directory, file_glob)
 
    summ = summary_table(data, initial_counts)
 
    print("=" * 92)
    print("PER-FUNCTION SUMMARY")
    print("=" * 92)
    hdr = (f"{'Func':<5}{'d':<3}{'n':<5}{'rounds':<8}{'Best Y':<13}"
           f"{'Found@rnd':<11}{'Improve vs init':<17}{'Rounds since best':<6}")
    print(hdr)
    print("-" * 92)
    for _, r in summ.iterrows():
        best_str = f"{r['best_Y']:.4g}"
        found = "initial" if r["best_found_round"] == 0 else f"R{int(r['best_found_round'])}"
        imp = "-" if np.isnan(r["improvement"]) else f"{r['improvement']:+.4g}"
        print(f"{int(r['func']):<5}{int(r['d']):<3}{int(r['n_total']):<5}"
              f"{int(r['n_rounds']):<8}{best_str:<13}{found:<11}{imp:<17}"
              f"{int(r['rounds_since_best'])}")
    print("=" * 92)
 
    log = new_max_log(data, initial_counts)
 
    print("\n" + "=" * 92)
    print("ROUND-BY-ROUND LOG  (only rounds that set a new maximum are starred)")
    print("=" * 92)
    for fid in sorted(data):
        sub = log[log["func"] == fid]
        new_maxes = sub[sub["new_max"] != ""]
        print(f"\nFunction {fid}  —  {len(new_maxes)} new max(es) across {len(sub)} rounds:")
        if len(new_maxes) == 0:
            print("   (no round beat the initial-data best)")
        for _, r in new_maxes.iterrows():
            print(f"   R{int(r['round']):<3} Y={r['Y']:<12.5g} gain={r['gain']:+.4g}   {r['x']}")
 
    return 
 
print_report(
    os.path.join(data_dir, "calibrated_data"),
    initial_counts=INITIAL_COUNTS,
)

In [ ]:
import importlib
import sys
print(sys.version.split()[0])   # e.g. 3.11.7
packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "scikit-learn": "sklearn",   # import name differs from package name
    "matplotlib": "matplotlib",
}

for pkg_name, import_name in packages.items():
    try:
        mod = importlib.import_module(import_name)
        print(f"{pkg_name}=={mod.__version__}")
    except ImportError:
        print(f"# {pkg_name} not installed")